In [1]:
import pandas as pd
from sklearn.calibration import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import joblib

df=pd.read_excel('Crop_recommendation.xlsx',engine='openpyxl')
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
X=df.drop(['label'],axis=1)
y=df['label']
label_encoder = LabelEncoder()
y= label_encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify=y)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
xgb = XGBClassifier(n_estimators=50, learning_rate=0.1, max_depth=5, use_label_encoder=False, eval_metric='mlogloss')
lr=LogisticRegression(max_iter=10000)
lr.fit(X_train, y_train)
rf.fit(X_train, y_train)
xgb.fit(X_train, y_train)


/home/linux/coding/venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/linux/coding/venv/lib/python3.13/site-packages/xgboost/training.py:183: UserWarning: [18:37:11] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=50, n_jobs=None,
              num_parallel_tree=None, ...)

In [3]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, log_loss
from sklearn.model_selection import cross_val_score

# Dictionary to hold final results
results = []

# Evaluate each model
for name, model in {
    "Logistic Regression": lr,
    "Random Forest": rf,
    "XGBoost": xgb
}.items():
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    ll = log_loss(y_test, y_proba)
    cv = cross_val_score(model, X, y, cv=5, scoring='accuracy').mean()
    
    results.append([name, acc, prec, rec, f1, cv, ll])

# Print neatly
print("\n== Model Evaluation Summary ==")
print(f"{'Model':<20} {'Acc':<6} {'Prec':<6} {'Recall':<6} {'F1':<6} {'CV':<6} {'LogLoss':<8}")
for r in results:
    print(f"{r[0]:<20} {r[1]:.4f} {r[2]:.4f} {r[3]:.4f} {r[4]:.4f} {r[5]:.4f} {r[6]:.4f}")


/home/linux/coding/venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/linux/coding/venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-


== Model Evaluation Summary ==
Model                Acc    Prec   Recall F1     CV     LogLoss 
Logistic Regression  0.9409 0.9416 0.9409 0.9409 0.9373 0.1467
Random Forest        0.9682 0.9691 0.9682 0.9678 0.9664 0.1181
XGBoost              0.9682 0.9685 0.9682 0.9680 0.9664 0.1199


In [4]:
print("\n== Model Evaluation Summary ==")
print(f"{'Model':<20} {'Acc':<6} {'Prec':<6} {'Recall':<6} {'F1':<6} {'CV':<6} {'LogLoss':<8}")
for r in results:
    print(f"{r[0]:<20} {r[1]:.4f} {r[2]:.4f} {r[3]:.4f} {r[4]:.4f} {r[5]:.4f} {r[6]:.4f}")


== Model Evaluation Summary ==
Model                Acc    Prec   Recall F1     CV     LogLoss 
Logistic Regression  0.9409 0.9416 0.9409 0.9409 0.9373 0.1467
Random Forest        0.9682 0.9691 0.9682 0.9678 0.9664 0.1181
XGBoost              0.9682 0.9685 0.9682 0.9680 0.9664 0.1199


In [6]:
import shap
import os

# Save models
os.makedirs('models', exist_ok=True)
joblib.dump(lr, 'models/logistic_model.pkl')
joblib.dump(rf, 'models/random_forest_model.pkl')
joblib.dump(xgb, 'models/xgb_model.pkl')

# Create and save SHAP explainers
os.makedirs('xai', exist_ok=True)
shap_explainer_lr = shap.Explainer(lr, X_train)
shap_explainer_rf = shap.Explainer(rf, X_train)
shap_explainer_xgb = shap.Explainer(xgb, X_train)

joblib.dump(shap_explainer_lr, 'xai/shap_logistic.pkl')
joblib.dump(shap_explainer_rf, 'xai/shap_rf.pkl')
joblib.dump(shap_explainer_xgb, 'xai/shap_xgb.pkl')

# Save label encoder
joblib.dump(label_encoder, 'models/label_encoder.pkl')


['models/label_encoder.pkl']

In [16]:
import shap
import pickle

# For logistic model
explainer_logistic = shap.Explainer(lr, X_train)
pickle.dump(explainer_logistic, open('shap_logistic.pkl', 'wb'))

# For xgboost model
explainer_xgb = shap.Explainer(xgb, X_train)
pickle.dump(explainer_xgb, open('shap_xgboost.pkl', 'wb'))
